# Real vs. AI-Generated Music Detection: Data Preparation

Builds the dataset used by the main pipeline (`real_vs_ai_music_detection.ipynb`):
- Downloads the SONICS real songs (YouTube IDs from the dataset) and a sample of
  AI-generated songs from the SONICS release on Hugging Face
- Builds the raw manifest, then a stratified song-level train/val/test split (70/15/15)
- Standardizes audio to 16 kHz mono and segments it into non-overlapping 10-second clips

SONICS dataset: https://github.com/awsaf49/sonics

**Running it:** built for Google Colab. Mount Google Drive and replace `PROJECT_ROOT`
in the paths with your Drive folder.

#### Installation of youtube downloader

In [ ]:
!pip install -q yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.5 MB/s eta 0:00:00


Test for youtube downloader (1 file)

In [ ]:
import os
import yt_dlp

def download_youtube_audio_wav(youtube_id_or_url: str, output_dir: str = "./audio_data") -> str:
    """
    Downloads audio from YouTube and converts it directly to a 16kHz mono WAV file
    using FFmpeg inside Google Colab.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Construct full YouTube URL if only ID is provided
    if not youtube_id_or_url.startswith("http"):
        url = f"https://www.youtube.com/watch?v={youtube_id_or_url}"
    else:
        url = youtube_id_or_url

    # Output template: saves as output_dir/{video_id}.wav
    out_template = os.path.join(output_dir, "%(id)s.%(ext)s")

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': out_template,
        'quiet': True,
        'no_warnings': True,
        # Force FFmpeg to extract audio as WAV
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
        }],
        # Target audio properties: 16kHz sample rate (-ar 16000), 1 channel / mono (-ac 1)
        'postprocessor_args': [
            '-ar', '16000',
            '-ac', '1'
        ],
        'prefer_ffmpeg': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(url, download=True)
        video_id = info_dict.get("id", "audio")
        expected_wav_path = os.path.join(output_dir, f"{video_id}.wav")

    return expected_wav_path


# Example usage over a list of YouTube IDs:
youtube_ids = ["dQw4w9WgXcQ"]  # Replace with your target IDs

for yt_id in youtube_ids:
    wav_path = download_youtube_audio_wav(yt_id, output_dir="./dataset_wavs")
    print(f"Saved 16kHz mono WAV to: {wav_path}")

Saved 16kHz mono WAV to: ./dataset_wavs/dQw4w9WgXcQ.wav


Short script for initial handling of Sonics dataset (real songs side), based on their Github.

Github: https://github.com/awsaf49/sonics


Kaggle (dataset): https://www.kaggle.com/datasets/awsaf49/sonics-dataset?hl=en-US

Updated description: **Download to drive dataset of 5K real songs from youtube. Running in batches to avoid drive/colab/youtube access issues**

In [ ]:
import os
import time
import random
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import yt_dlp
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')
# ==========================================
# CONFIGURATION & PATHS
# ==========================================
BASE_PATH = "/content/drive/MyDrive/PROJECT_ROOT/datasets"
CSV_PATH = os.path.join(BASE_PATH, "real_songs.csv")
OUTPUT_DIR = os.path.join(BASE_PATH, "real_songs/")
FAILED_LOG_PATH = os.path.join(BASE_PATH, "failed_youtube_ids.txt")

TARGET_SONGS = 5000
BATCH_SIZE = 50
MAX_WORKERS = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_target_filename(row):
    """Helper to determine the exact filename this row will produce."""
    yt_id = str(row['youtube_id']).strip()
    if 'filename' in row and pd.notna(row['filename']):
        base_name = os.path.splitext(str(row['filename']).strip())[0]
        return f"{base_name}.wav"
    return f"{yt_id}.wav"

def download_single_song(row):
    """Downloads, converts to 16kHz mono WAV, and handles errors."""
    time.sleep(random.uniform(1.5, 3.5))

    yt_id = str(row['youtube_id']).strip()
    target_filename = row['target_file']
    out_path = os.path.join(OUTPUT_DIR, target_filename)

    url = f"https://www.youtube.com/watch?v={yt_id}"

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': os.path.splitext(out_path)[0],
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
        }],
        'postprocessor_args': ['-ar', '16000', '-ac', '1'],
        'prefer_ffmpeg': True,
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': False,
        'socket_timeout': 15,
        'retries': 3,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return {"id": yt_id, "status": "success", "error": None}
    except Exception as e:
        if os.path.exists(out_path):
            try:
                os.remove(out_path)
            except OSError:
                pass
        return {"id": yt_id, "status": "failed", "error": str(e)}

def main():
    if not os.path.exists(CSV_PATH):
        print(f"[ERROR] CSV not found at: {CSV_PATH}")
        return

    # 1. Scan output directory for existing files
    print("Scanning Drive for existing downloaded songs...")
    existing_files = set(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.wav'))
    current_count = len(existing_files)

    print(f"Found {current_count} songs already downloaded.")
    if current_count >= TARGET_SONGS:
        print(f"Goal of {TARGET_SONGS} songs reached!")
        return

    # 2. Load and Shuffle Data for Randomness
    df = pd.read_csv(CSV_PATH)
    # Shuffle the entire dataframe to ensure random selection
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    df['target_file'] = df.apply(get_target_filename, axis=1)

    # 3. Load failed IDs
    failed_ids = set()
    if os.path.exists(FAILED_LOG_PATH):
        with open(FAILED_LOG_PATH, 'r', encoding='utf-8') as f:
            failed_ids = set(line.strip().split('\t')[0] for line in f if line.strip())

    # 4. Filter
    df_filtered = df[~df['target_file'].isin(existing_files)]
    df_filtered = df_filtered[~df_filtered['youtube_id'].isin(failed_ids)]

    if df_filtered.empty:
        print("No more songs to download!")
        return

    # 5. Take batch
    songs_needed = TARGET_SONGS - current_count
    current_batch_size = min(BATCH_SIZE, songs_needed)
    batch_df = df_filtered.head(current_batch_size)

    print(f"Starting random batch of {len(batch_df)} songs...")

    results = {"success": 0, "failed": 0}
    new_failed_ids = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(download_single_song, row): row for _, row in batch_df.iterrows()}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Batch Progress"):
            res = future.result()
            results[res["status"]] += 1
            if res["status"] == "failed":
                new_failed_ids.append((res["id"], res["error"]))

    print(f"\nTotal now on Drive: {len(os.listdir(OUTPUT_DIR))} / {TARGET_SONGS}")

    if new_failed_ids:
        with open(FAILED_LOG_PATH, "a", encoding="utf-8") as f:
            for y_id, err in new_failed_ids:
                f.write(f"{y_id}\t{str(err).strip()}\n")

if __name__ == "__main__":
    main()

Mounted at /content/drive
Scanning Drive for existing downloaded songs...
Found 5000 songs already downloaded.
Goal of 5000 songs reached!


#### Download to drive a random sample of 5K AI-generated (fake) songs from SONICS.
Extracted from zip parts on Hugging Face since audio isn't stored per-file, unlike the real songs.

In [ ]:
!pip install -q huggingface_hub
!apt-get -qq install -y ffmpeg > /dev/null

In [ ]:
import os
import random
import zipfile
import shutil
import subprocess
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
BASE_PATH = "/content/drive/MyDrive/PROJECT_ROOT/datasets"
FAKE_OUTPUT_DIR = os.path.join(BASE_PATH, "fake_songs")
os.makedirs(FAKE_OUTPUT_DIR, exist_ok=True)

TARGET_FAKE_SONGS = 5000
REPO_ID = "awsaf49/sonics"

def extract_and_convert(zf, member_path, out_path):
    """Extract one mp3 from an open zip, convert to 16kHz mono WAV, save to out_path."""
    tmp_mp3 = "/tmp/_tmp_fake.mp3"
    with zf.open(member_path) as src, open(tmp_mp3, 'wb') as dst:
        shutil.copyfileobj(src, dst)
    subprocess.run(
        ["ffmpeg", "-y", "-i", tmp_mp3, "-ar", "16000", "-ac", "1", out_path],
        check=True, capture_output=True
    )
    os.remove(tmp_mp3)

def main():
    # 1. Scan Drive for existing files (idempotent, same as real_songs script)
    print("Scanning Drive for existing downloaded fake songs...")
    existing = set(os.listdir(FAKE_OUTPUT_DIR))
    print(f"Found {len(existing)} fake songs already downloaded.")
    if len(existing) >= TARGET_FAKE_SONGS:
        print(f"Goal of {TARGET_FAKE_SONGS} songs reached!")
        return

    # 2. Load metadata, sample what's still needed
    csv_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename="fake_songs.csv")
    df = pd.read_csv(csv_path, low_memory=False)
    df["mp3_name"] = df["filename"].astype(str) + ".mp3"
    df["wav_name"] = df["filename"].astype(str) + ".wav"
    df = df[~df["wav_name"].isin(existing)]

    random.seed(42)
    n_needed = min(TARGET_FAKE_SONGS - len(existing), len(df))
    sample_df = df.sample(n=n_needed, random_state=42)
    wanted = dict(zip(sample_df["mp3_name"], sample_df["wav_name"]))
    print(f"Need {len(wanted)} more fake songs to hit {TARGET_FAKE_SONGS}.")

    # 3. Pull zip parts one at a time, extract + convert only what's needed
    copied = 0
    failed = []
    part_num = 1

    while wanted and copied < len(sample_df):
        part_name = f"fake_songs/part_{part_num:02d}.zip"
        try:
            zip_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=part_name)
        except Exception as e:
            print(f"No more parts found (stopped at part_{part_num:02d}): {e}")
            break

        with zipfile.ZipFile(zip_path, 'r') as zf:
            names_in_zip = {os.path.basename(n): n for n in zf.namelist()}
            matches = set(wanted.keys()) & set(names_in_zip.keys())

            for mp3_name in matches:
                wav_name = wanted[mp3_name]
                out_path = os.path.join(FAKE_OUTPUT_DIR, wav_name)
                try:
                    extract_and_convert(zf, names_in_zip[mp3_name], out_path)
                    copied += 1
                except subprocess.CalledProcessError as e:
                    failed.append((mp3_name, str(e)))
                del wanted[mp3_name]

        os.remove(zip_path)
        print(f"After part_{part_num:02d}: copied {copied}/{len(sample_df)} so far.")
        part_num += 1

    print(f"\nTotal now on Drive: {len(os.listdir(FAKE_OUTPUT_DIR))} / {TARGET_FAKE_SONGS}")
    if failed:
        print(f"{len(failed)} conversion failures logged.")
        for mp3_name, err in failed:
            print(f"  {mp3_name}: {err}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scanning Drive for existing downloaded fake songs...
Found 4879 fake songs already downloaded.


KeyboardInterrupt: 

#### Build the manifest

In [ ]:
!pip install -q soundfile
!pip install -q tqdm
from tqdm import tqdm
import os
import soundfile as sf
import pandas as pd
from google.colab import drive
import time

In [ ]:
!pip install -q soundfile tqdm

In [ ]:
import os
import time
from tqdm import tqdm
import soundfile as sf
import pandas as pd
from google.colab import drive

print("Starting cell, mounting Drive...", flush=True)
drive.mount('/content/drive', force_remount=True)
print("Drive mounted.", flush=True)

BASE_PATH = "/content/drive/MyDrive/PROJECT_ROOT/datasets"
REAL_DIR = os.path.join(BASE_PATH, "real_songs")
FAKE_DIR = os.path.join(BASE_PATH, "fake_songs")
MANIFEST_PATH = os.path.join(BASE_PATH, "raw_manifest.csv")

# Load existing manifest if present, so we know what's already been scanned
if os.path.exists(MANIFEST_PATH):
    existing_df = pd.read_csv(MANIFEST_PATH)
    existing_paths = set(existing_df["filepath"])
    print(f"Loaded existing manifest: {len(existing_df)} rows.")
else:
    existing_df = pd.DataFrame(columns=["filepath", "source", "label", "duration"])
    existing_paths = set()
    print("No existing manifest found, starting fresh.")

def scan_folder(folder_path, source, label, existing_paths, batch_pause_every=200, pause_secs=1.0):
    rows = []
    skipped = []
    all_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.wav', '.flac'))]
    files = [f for f in all_files if os.path.join(folder_path, f) not in existing_paths]
    print(f"  {len(all_files)} files on disk, {len(files)} new (skipping {len(all_files) - len(files)} already in manifest).")

    if not files:
        print("  Nothing new to scan.")
        return rows

    for i, fname in enumerate(tqdm(files, desc=f"Scanning {os.path.basename(folder_path)}")):
        full_path = os.path.join(folder_path, fname)
        success = False
        last_err = None
        for attempt in range(3):
            try:
                info = sf.info(full_path)
                duration = info.frames / info.samplerate
                rows.append({"filepath": full_path, "source": source, "label": label, "duration": duration})
                success = True
                break
            except Exception as e:
                last_err = str(e)
                time.sleep(0.5)
        if not success:
            skipped.append((fname, last_err))
            print(f"  [FAILED] {fname}: {last_err}")

        if (i + 1) % batch_pause_every == 0:
            print(f"  ...{i + 1}/{len(files)} scanned ({len(rows)} ok, {len(skipped)} failed so far)")
            time.sleep(pause_secs)

    print(f"  Done: {len(rows)} succeeded, {len(skipped)} failed out of {len(files)} new files.")
    if skipped:
        print(f"  Skipped {len(skipped)} unreadable files after retries:")
        for fname, err in skipped[:5]:
            print(f"    {fname}: {err}")
    return rows

print("Scanning real_songs/ ...")
real_rows = scan_folder(REAL_DIR, source="sonics", label="real", existing_paths=existing_paths)
print(f"  {len(real_rows)} new real songs added.\n")

print("Scanning fake_songs/ ...")
fake_rows = scan_folder(FAKE_DIR, source="sonics", label="fake", existing_paths=existing_paths)
print(f"  {len(fake_rows)} new fake songs added.\n")

new_df = pd.DataFrame(real_rows + fake_rows)
manifest_df = pd.concat([existing_df, new_df], ignore_index=True)

before = len(manifest_df)
manifest_df = manifest_df.drop_duplicates(subset="filepath", keep="first")
if len(manifest_df) < before:
    print(f"Dropped {before - len(manifest_df)} duplicate filepath rows.")

manifest_df.to_csv(MANIFEST_PATH, index=False)

print(f"\nManifest saved to: {MANIFEST_PATH}")
print(f"Total: {len(manifest_df)} clips ({(manifest_df['label']=='real').sum()} real, {(manifest_df['label']=='fake').sum()} fake)")
print(f"\nLabel breakdown:\n{manifest_df['label'].value_counts()}")

Starting cell, mounting Drive...
Mounted at /content/drive
Drive mounted.
Loaded existing manifest: 9879 rows.
Scanning real_songs/ ...
  5000 files on disk, 0 new (skipping 5000 already in manifest).
  Nothing new to scan.
  0 new real songs added.

Scanning fake_songs/ ...
  4879 files on disk, 0 new (skipping 4879 already in manifest).
  Nothing new to scan.
  0 new fake songs added.


Manifest saved to: /content/drive/MyDrive/PROJECT_ROOT/datasets/raw_manifest.csv
Total: 9879 clips (5000 real, 4879 fake)

Label breakdown:
label
real    5000
fake    4879
Name: count, dtype: int64


In [ ]:
import os

SRC_MODULES = [
    "data_acquisition", "preprocessing", "dataset",
    "models", "training", "evaluation", "utils"
]

PROJECT_ROOT = "/content/drive/MyDrive/PROJECT_ROOT"
for mod in SRC_MODULES:
    path = os.path.join(PROJECT_ROOT, "src", mod)
    os.makedirs(path, exist_ok=True)
    init_file = os.path.join(path, "__init__.py")
    if not os.path.exists(init_file):
        open(init_file, "a").close()

os.makedirs(os.path.join(PROJECT_ROOT, "configs"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, "notebooks"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, "results"), exist_ok=True)

print("Structure created under", PROJECT_ROOT)

Structure created under /content/drive/MyDrive/PROJECT_ROOT


## Block 2: Preprocessing — Item-Level Split

Input: `raw_manifest.csv` from Block 1 (one row per song, not yet segmented into clips).

Before any 10-second clip segmentation happens, we split at the **item level**
(whole songs), stratified by label, using a fixed seed for reproducibility. This
guarantees that when a song is later cut into multiple clips, all of its clips
land in the same partition — preventing near-duplicate segments from leaking
across train/val/test.

Split: 70% train / 15% validation / 15% test.
- **Train**: model fitting + augmentation
- **Validation**: early stopping, checkpoint selection, hyperparameter selection, threshold selection
- **Test**: untouched until final reporting

Output: the raw manifest with a new `split` column, saved as `item_split_manifest.csv`.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

BASE_PATH = "/content/drive/MyDrive/PROJECT_ROOT/datasets"
RAW_MANIFEST_PATH = os.path.join(BASE_PATH, "raw_manifest.csv")
ITEM_SPLIT_MANIFEST_PATH = os.path.join(BASE_PATH, "item_split_manifest.csv")

SPLIT_SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

print(os.path.exists(RAW_MANIFEST_PATH), RAW_MANIFEST_PATH)
raw_df = pd.read_csv(RAW_MANIFEST_PATH)
print(f"Loaded {len(raw_df)} items.")
print(raw_df['label'].value_counts())

# First split off train, then split the remainder into val/test
train_df, rest_df = train_test_split(
    raw_df,
    train_size=TRAIN_FRAC,
    stratify=raw_df['label'],
    random_state=SPLIT_SEED,
)

# rest_df is 30% of data; val/test should be 15/15 of the total,
# i.e. 50/50 of rest_df
val_df, test_df = train_test_split(
    rest_df,
    train_size=VAL_FRAC / (VAL_FRAC + TEST_FRAC),
    stratify=rest_df['label'],
    random_state=SPLIT_SEED,
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

item_split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
item_split_df.to_csv(ITEM_SPLIT_MANIFEST_PATH, index=False)

print(f"\nSaved item-level split manifest to: {ITEM_SPLIT_MANIFEST_PATH}")
print(f"\nSplit sizes:\n{item_split_df['split'].value_counts()}")
print(f"\nSplit x label breakdown:\n{item_split_df.groupby(['split', 'label']).size()}")

## Block 2: Preprocessing — Segmentation

Input: `item_split_manifest.csv` (song-level, with `split` already assigned).

For each song, load the audio, standardize it, and cut it into non-overlapping
10-second clips. This is the deterministic part of preprocessing — no augmentation
here, since augmentation is applied on the fly inside the Dataset class (Block 3),
only to the SONICS training loader.

Standardization target (per architecture plan):
- 16 kHz
- mono
- 10-second clips
- WAV format
- clip filenames preserve the source item ID + clip index

Output: clips under `data/processed/<source>/<split>/<label>/`, plus a new
**clip-level manifest** (`clip_manifest.csv`) with one row per clip: filepath,
parent item ID, source, label, split, clip index, duration.

This step is resumable: if a song's clips already exist on disk, it's skipped,
so reruns after a disconnect only process what's missing.

In [ ]:
!pip install -q librosa soundfile tqdm

import os
import time
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm

BASE_PATH = "/content/drive/MyDrive/PROJECT_ROOT/datasets"
ITEM_SPLIT_MANIFEST_PATH = os.path.join(BASE_PATH, "item_split_manifest.csv")
PROCESSED_DIR = os.path.join(BASE_PATH, "processed")
CLIP_MANIFEST_PATH = os.path.join(BASE_PATH, "clip_manifest.csv")

TARGET_SR = 16000
CLIP_SECONDS = 10
CLIP_SAMPLES = TARGET_SR * CLIP_SECONDS

item_split_df = pd.read_csv(ITEM_SPLIT_MANIFEST_PATH)
print(f"Loaded {len(item_split_df)} items.")
print(item_split_df.groupby(['split', 'label']).size())

# Load existing clip manifest if present, so reruns skip already-processed items
if os.path.exists(CLIP_MANIFEST_PATH):
    existing_clip_df = pd.read_csv(CLIP_MANIFEST_PATH)
    processed_item_ids = set(existing_clip_df['item_filepath'])
    print(f"Loaded existing clip manifest: {len(existing_clip_df)} clips from {len(processed_item_ids)} items.")
else:
    existing_clip_df = pd.DataFrame(columns=[
        "clip_filepath", "item_filepath", "source", "label", "split", "clip_index", "duration"
    ])
    processed_item_ids = set()
    print("No existing clip manifest found, starting fresh.")

items_to_process = item_split_df[~item_split_df['filepath'].isin(processed_item_ids)]
print(f"\n{len(items_to_process)} items still need segmentation "
      f"(skipping {len(item_split_df) - len(items_to_process)} already processed).")

def segment_item(row):
    """Load one song, resample to 16kHz mono, cut into 10-sec clips, save each."""
    src_path = row['filepath']
    source = row['source']
    label = row['label']
    split = row['split']
    item_id = os.path.splitext(os.path.basename(src_path))[0]

    out_dir = os.path.join(PROCESSED_DIR, source, split, label)
    os.makedirs(out_dir, exist_ok=True)

    clip_rows = []
    try:
        audio, sr = librosa.load(src_path, sr=TARGET_SR, mono=True)
    except Exception as e:
        return [], (src_path, str(e))

    n_clips = len(audio) // CLIP_SAMPLES  # drop any final partial clip
    for i in range(n_clips):
        start = i * CLIP_SAMPLES
        end = start + CLIP_SAMPLES
        clip_audio = audio[start:end]

        clip_fname = f"{item_id}_clip{i:03d}.wav"
        clip_path = os.path.join(out_dir, clip_fname)

        if not os.path.exists(clip_path):
            sf.write(clip_path, clip_audio, TARGET_SR)

        clip_rows.append({
            "clip_filepath": clip_path,
            "item_filepath": src_path,
            "source": source,
            "label": label,
            "split": split,
            "clip_index": i,
            "duration": CLIP_SECONDS,
        })

    return clip_rows, None

all_new_rows = []
failed_items = []

pbar = tqdm(items_to_process.itertuples(index=False), total=len(items_to_process), desc="Segmenting")
for row in pbar:
    row_dict = row._asdict()
    clip_rows, error = segment_item(row_dict)
    if error:
        failed_items.append(error)
    else:
        all_new_rows.extend(clip_rows)
    pbar.set_postfix(clips=len(all_new_rows), failed=len(failed_items))

if failed_items:
    print(f"\n{len(failed_items)} items failed to load:")
    for path, err in failed_items[:5]:
        print(f"  {path}: {err}")

new_clip_df = pd.DataFrame(all_new_rows)
clip_manifest_df = pd.concat([existing_clip_df, new_clip_df], ignore_index=True)
clip_manifest_df = clip_manifest_df.drop_duplicates(subset="clip_filepath", keep="first")

clip_manifest_df.to_csv(CLIP_MANIFEST_PATH, index=False)

print(f"\nClip manifest saved to: {CLIP_MANIFEST_PATH}")
print(f"Total clips: {len(clip_manifest_df)}")
print(f"\nSplit x label breakdown:\n{clip_manifest_df.groupby(['split', 'label']).size()}")